In [6]:
!pip install scikit-learn chardet PyPDF2

import os
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import chardet
from PyPDF2 import PdfReader  # PDF handling

class TfidfSearchModel:
    def __init__(self):
        """
        Initialize the TF-IDF Search Model.
        """
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None

    def fit_tfidf(self, documents):
        """
        Train the TF-IDF vectorizer and generate the TF-IDF matrix.
        """
        self.tfidf_vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(documents)
        print("TF-IDF training complete.")

    def save_tfidf(self, vectorizer_path, matrix_path):
        """
        Save the TF-IDF vectorizer and matrix to files.
        """
        with open(vectorizer_path, "wb") as f:
            pickle.dump(self.tfidf_vectorizer, f)
        with open(matrix_path, "wb") as f:
            pickle.dump(self.tfidf_matrix, f)
        print("TF-IDF vectorizer and matrix saved.")

    def load_tfidf(self, vectorizer_path, matrix_path):
        """
        Load the TF-IDF vectorizer and matrix from files.
        """
        with open(vectorizer_path, "rb") as f:
            self.tfidf_vectorizer = pickle.load(f)
        with open(matrix_path, "rb") as f:
            self.tfidf_matrix = pickle.load(f)
        print("TF-IDF vectorizer and matrix loaded.")

    def search(self, query, documents, top_k=5):
        """
        Perform a search using only TF-IDF.
        """
        if self.tfidf_vectorizer is None or self.tfidf_matrix is None:
            raise ValueError("TF-IDF model has not been trained or loaded.")

        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        top_indices = np.argsort(tfidf_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents



def read_pdf(file_path):
    """
    Extract text content from a PDF file.
    """
    try:
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() for page in reader.pages if page.extract_text())
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from PDF {file_path}: {e}")
        return None


if __name__ == "__main__":
    # Path to folder containing document files
    folder_path = "/content/drive/MyDrive/documents/"  # Replace with your folder path
    file_names = os.listdir(folder_path)

    # Load all documents from the folder
    documents = []
    doc_names = []
    for file_name in file_names:
        file_path = os.path.join(folder_path, file_name)
        try:
            # Handle PDF files
            if file_name.lower().endswith(".pdf"):
                content = read_pdf(file_path)
            else:
                # Detect encoding of non-PDF files
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']
                    if not encoding:
                        raise UnicodeDecodeError(f"Encoding not detected for {file_name}.")

                # Read the file with detected encoding
                with open(file_path, 'r', encoding=encoding) as file:
                    content = file.read()

            if content:
                documents.append(content)
                doc_names.append(file_name)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

    print(f"Loaded {len(documents)} documents.")

    # Initialize the TF-IDF Search Model
    model = TfidfSearchModel()

    # Train TF-IDF on documents
    model.fit_tfidf(documents)
    model.save_tfidf("tfidf_vectorizer_only.pkl", "tfidf_matrix_only.pkl")

    # Load saved TF-IDF components (optional)
    model.load_tfidf("tfidf_vectorizer_only.pkl", "tfidf_matrix_only.pkl")

    # Get input query from the user
    query = input("Enter your search query: ")

    # Perform TF-IDF search
    top_k = 5
    ranked_documents = model.search(query, documents, top_k=top_k)

    # Display ranked results
    print("\nRanked Documents (TF-IDF Only):")
    for idx, doc in enumerate(ranked_documents, 1):
        print(f"{idx}: {doc_names[documents.index(doc)]}")

    # Optionally save results
    with open('ranked_results_tfidf_only.txt', 'w') as output_file:
        for idx, doc in enumerate(ranked_documents, 1):
            output_file.write(f"{idx}: {doc_names[documents.index(doc)]}\n")
    print("\nRanked results (TF-IDF only) saved to 'ranked_results_tfidf_only.txt'.")

Loaded 8 documents.
TF-IDF training complete.
TF-IDF vectorizer and matrix saved.
TF-IDF vectorizer and matrix loaded.
Enter your search query: price

Ranked Documents (TF-IDF Only):
1: s00181-024-02644-6.pdf
2: 1-s2.0-S0040162524006449-main.pdf
3: 2501.01010v2.pdf
4: 1-s2.0-S0020025524015652-main.pdf
5: 1-s2.0-S1568494624014224-main.pdf

Ranked results (TF-IDF only) saved to 'ranked_results_tfidf_only.txt'.


In [9]:
!pip install torch transformers scikit-learn chardet PyPDF2

import os
import chardet
from PyPDF2 import PdfReader
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

class BertSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased"):
        """
        Initialize the BERT-only Search Model.
        """
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()  # Use GPU if available

    def encode_batch(self, texts, batch_size=16):
        """
        Encode a batch of texts using the BERT model and return sentence embeddings.
        """
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)  # Normalize for cosine similarity

    def search(self, query, documents, top_k=5):
        """
        Perform a search using only BERT embeddings.
        """
        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(documents)
        similarity_scores = cosine_similarity(query_embedding.detach().cpu().numpy(), doc_embeddings.detach().cpu().numpy())[0]
        top_indices = np.argsort(similarity_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents

def read_pdf(file_path):
    """
    Extract text content from a PDF file.
    """
    try:
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() for page in reader.pages if page.extract_text())
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from PDF {file_path}: {e}")
        return None

if __name__ == "__main__":
    # Path to folder containing document files
    folder_path = "/content/drive/MyDrive/documents/"  # Replace with your folder path
    file_names = os.listdir(folder_path)

    # Load all documents from the folder
    documents = []
    doc_names = []
    for file_name in file_names:
        file_path = os.path.join(folder_path, file_name)
        try:
            # Handle PDF files
            if file_name.lower().endswith(".pdf"):
                content = read_pdf(file_path)
            else:
                # Detect encoding of non-PDF files
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']
                    if not encoding:
                        raise UnicodeDecodeError(f"Encoding not detected for {file_name}.")

                # Read the file with detected encoding
                with open(file_path, 'r', encoding=encoding) as file:
                    content = file.read()

            if content:
                documents.append(content)
                doc_names.append(file_name)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

    print(f"Loaded {len(documents)} documents.")

    # Initialize the BERT-only Search Model
    model = BertSearchModel(bert_model_name="bert-base-uncased")

    # Get input query from the user
    query = input("Enter your search query: ")

    # Perform BERT-only search
    top_k = 5
    ranked_documents = model.search(query, documents, top_k=top_k)

    # Display ranked results
    print("\nRanked Documents (BERT Only):")
    for idx, doc in enumerate(ranked_documents, 1):
        print(f"{idx}: {doc_names[documents.index(doc)]}")

    # Optionally save results
    with open('ranked_results_bert_only.txt', 'w') as output_file:
        for idx, doc in enumerate(ranked_documents, 1):
            output_file.write(f"{idx}: {doc_names[documents.index(doc)]}\n")
    print("\nRanked results (BERT only) saved to 'ranked_results_bert_only.txt'.")

Loaded 8 documents.
Enter your search query: price

Ranked Documents (BERT Only):
1: 1333-Article Text-11950-1-10-20250118.pdf
2: pediatrics-2025-1-e64814.pdf
3: 1-s2.0-S0040162524006449-main.pdf
4: hoffman-kaplan-the-incidence-of-congenital-heart-disease.pdf
5: 1-s2.0-S0020025524015652-main.pdf

Ranked results (BERT only) saved to 'ranked_results_bert_only.txt'.


In [5]:
!pip install torch transformers scikit-learn chardet PyPDF2

import os
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import chardet
from PyPDF2 import PdfReader  # PDF handling

class HybridSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased"):
        """
        Initialize the Hybrid Search Model with BERT and TF-IDF
        """
        # Initialize BERT Model and Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()  # Use GPU
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None

    def fit_tfidf(self, documents):
        """
        Train the TF-IDF vectorizer and generate the TF-IDF matrix.
        """
        self.tfidf_vectorizer = TfidfVectorizer()
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(documents)
        print("TF-IDF training complete.")

    def save_tfidf(self, vectorizer_path, matrix_path):
        """
        Save the TF-IDF vectorizer and matrix to files.
        """
        with open(vectorizer_path, "wb") as f:
            pickle.dump(self.tfidf_vectorizer, f)
        with open(matrix_path, "wb") as f:
            pickle.dump(self.tfidf_matrix, f)
        print("TF-IDF vectorizer and matrix saved.")

    def load_tfidf(self, vectorizer_path, matrix_path):
        """
        Load the TF-IDF vectorizer and matrix from files.
        """
        with open(vectorizer_path, "rb") as f:
            self.tfidf_vectorizer = pickle.load(f)
        with open(matrix_path, "rb") as f:
            self.tfidf_matrix = pickle.load(f)
        print("TF-IDF vectorizer and matrix loaded.")

    def encode_batch(self, texts, batch_size=16):
        """
        Encode a batch of texts using the BERT model and return sentence embeddings.
        """
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)  # Normalize for cosine similarity

    def hybrid_search(self, query, documents, top_k=5, tfidf_weight=0.4):
        """
        Perform a hybrid search with TF-IDF filtering and BERT semantic ranking.
        """
        # Step 1: TF-IDF Filtering
        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        tfidf_top_indices = np.argsort(tfidf_scores)[-2 * top_k:][::-1]  # Take more docs for BERT filtering
        filtered_docs = [documents[i] for i in tfidf_top_indices]

        # Step 2: BERT Semantic Ranking
        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(filtered_docs)
        bert_scores = torch.nn.functional.cosine_similarity(query_embedding, doc_embeddings).cpu().detach().numpy()

        # Step 3: Combine Scores
        combined_scores = (
            tfidf_weight * tfidf_scores[tfidf_top_indices] + (1 - tfidf_weight) * bert_scores
        )
        final_indices = np.argsort(combined_scores)[-top_k:][::-1]  # Final ranking
        final_ranked_docs = [filtered_docs[i] for i in final_indices]

        return final_ranked_docs



def read_pdf(file_path):
    """
    Extract text content from a PDF file.
    """
    try:
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() for page in reader.pages if page.extract_text())
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from PDF {file_path}: {e}")
        return None


if __name__ == "__main__":
    # Path to folder containing document files
    folder_path = "/content/drive/MyDrive/documents/"  # Replace with your folder path
    file_names = os.listdir(folder_path)

    # Load all documents from the folder
    documents = []
    doc_names = []
    for file_name in file_names:
        file_path = os.path.join(folder_path, file_name)
        try:
            # Handle PDF files
            if file_name.lower().endswith(".pdf"):
                content = read_pdf(file_path)
            else:
                # Detect encoding of non-PDF files
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']
                    if not encoding:
                        raise UnicodeDecodeError(f"Encoding not detected for {file_name}.")

                # Read the file with detected encoding
                with open(file_path, 'r', encoding=encoding) as file:
                    content = file.read()

            if content:
                documents.append(content)
                doc_names.append(file_name)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

    print(f"Loaded {len(documents)} documents.")

    # Initialize the Hybrid Search Model
    model = HybridSearchModel(bert_model_name="bert-base-uncased")

    # Train TF-IDF on documents
    model.fit_tfidf(documents)
    model.save_tfidf("tfidf_vectorizer.pkl", "tfidf_matrix.pkl")

    # Load saved TF-IDF components (optional)
    model.load_tfidf("tfidf_vectorizer.pkl", "tfidf_matrix.pkl")

    # Get input query from the user
    query = input("Enter your search query: ")

    # Perform hybrid search
    top_k = 5
    ranked_documents = model.hybrid_search(query, documents, top_k=top_k, tfidf_weight=0.5)

    # Display ranked results
    print("\nRanked Documents:")
    for idx, doc in enumerate(ranked_documents, 1):
        print(f"{idx}: {doc_names[documents.index(doc)]}")

    # Optionally save results
    with open('ranked_results.txt', 'w') as output_file:
        for idx, doc in enumerate(ranked_documents, 1):
            output_file.write(f"{idx}: {doc_names[documents.index(doc)]}\n")
    print("\nRanked results saved to 'ranked_results.txt'.")


Loaded 8 documents.
TF-IDF training complete.
TF-IDF vectorizer and matrix saved.
TF-IDF vectorizer and matrix loaded.
Enter your search query: price

Ranked Documents:
1: s00181-024-02644-6.pdf
2: 1-s2.0-S0040162524006449-main.pdf
3: 1-s2.0-S0020025524015652-main.pdf
4: 2501.01010v2.pdf
5: 1333-Article Text-11950-1-10-20250118.pdf

Ranked results saved to 'ranked_results.txt'.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import time

# Compare Results
def compare_models(query, documents):
    print("\nTF-IDF Model:")
    start_time = time.time()
    tfidf_results, tfidf_scores = tfidf_model.search(query)
    print("Results:", [documents[i] for i in tfidf_results])
    print("Scores:", tfidf_scores)
    print("Time Taken:", time.time() - start_time, "seconds")

    print("\nBERT Model:")
    start_time = time.time()
    bert_results, bert_scores = bert_model.search(query, documents)
    print("Results:", [documents[i] for i in bert_results])
    print("Scores:", bert_scores.tolist())
    print("Time Taken:", time.time() - start_time, "seconds")

    print("\nHybrid Model:")
    start_time = time.time()
    # Ensure the Hybrid Model is properly trained or loaded
    hybrid_model.fit_tfidf(documents)  # Train the TF-IDF vectorizer
    hybrid_results = hybrid_model.search(query, documents, top_k=5)  # Perform the hybrid search

    print("Results:", hybrid_results)
    print("Time Taken:", time.time() - start_time, "seconds")

# Run comparison
compare_models(query, documents)



TF-IDF Model:
Results: ['Cats and dogs are friendly animals.', 'The mat is cozy for cats to sleep on.', 'Foggy mornings are perfect for a walk with pets.', 'Dogs are in the fog.', 'The cat is on the mat.']
Scores: [0.58952182 0.19567027 0.         0.         0.        ]
Time Taken: 0.01025247573852539 seconds

BERT Model:
Results: ['Dogs are in the fog.', 'The mat is cozy for cats to sleep on.', 'Cats and dogs are friendly animals.', 'The cat is on the mat.', 'Foggy mornings are perfect for a walk with pets.']
Scores: [0.6901576519012451, 0.6512720584869385, 0.6338626742362976, 0.6149616241455078, 0.5672008991241455]
Time Taken: 1.2328789234161377 seconds

Hybrid Model:
TF-IDF training complete.
Results: ['Dogs are in the fog.', 'The mat is cozy for cats to sleep on.', 'Cats and dogs are friendly animals.', 'The cat is on the mat.', 'Foggy mornings are perfect for a walk with pets.']
Time Taken: 0.5967674255371094 seconds


In [10]:
!pip install torch transformers scikit-learn chardet PyPDF2

import os
import pickle
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import chardet
from PyPDF2 import PdfReader  # PDF handling

class HybridSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased", tfidf_vectorizer_path="hybrid_tfidf_vectorizer.pkl", tfidf_matrix_path="hybrid_tfidf_matrix.pkl"):
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        self.tfidf_vectorizer_path = tfidf_vectorizer_path
        self.tfidf_matrix_path = tfidf_matrix_path
        self._load_tfidf()

    def _load_tfidf(self):
        try:
            with open(self.tfidf_vectorizer_path, "rb") as f:
                self.tfidf_vectorizer = pickle.load(f)
            with open(self.tfidf_matrix_path, "rb") as f:
                self.tfidf_matrix = pickle.load(f)
            print("Hybrid Search TF-IDF vectorizer and matrix loaded.")
        except FileNotFoundError:
            print("Hybrid Search TF-IDF files not found. Please run the training script first.")
            self.tfidf_vectorizer = None
            self.tfidf_matrix = None

    def encode_batch(self, texts, batch_size=16):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)

    def hybrid_search(self, query, documents, top_k=5, tfidf_weight=0.4):
        if self.tfidf_vectorizer is None or self.tfidf_matrix is None:
            print("TF-IDF components not loaded. Hybrid search cannot be performed.")
            return []
        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        tfidf_top_indices = np.argsort(tfidf_scores)[-2 * top_k:][::-1]
        filtered_docs = [documents[i] for i in tfidf_top_indices]

        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(filtered_docs)
        bert_scores = torch.nn.functional.cosine_similarity(query_embedding, doc_embeddings).cpu().detach().numpy()

        combined_scores = (
            tfidf_weight * tfidf_scores[tfidf_top_indices] + (1 - tfidf_weight) * bert_scores
        )
        final_indices = np.argsort(combined_scores)[-top_k:][::-1]
        final_ranked_docs = [filtered_docs[i] for i in final_indices]
        return final_ranked_docs

class TfidfSearchModel:
    def __init__(self, vectorizer_path="tfidf_only_vectorizer.pkl", matrix_path="tfidf_only_matrix.pkl"):
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        self.vectorizer_path = vectorizer_path
        self.matrix_path = matrix_path
        self._load_tfidf()

    def _load_tfidf(self):
        try:
            with open(self.vectorizer_path, "rb") as f:
                self.tfidf_vectorizer = pickle.load(f)
            with open(self.matrix_path, "rb") as f:
                self.tfidf_matrix = pickle.load(f)
            print("TF-IDF Only vectorizer and matrix loaded.")
        except FileNotFoundError:
            print("TF-IDF Only files not found. Please run the training script first.")
            self.tfidf_vectorizer = None
            self.tfidf_matrix = None

    def search(self, query, documents, top_k=5):
        if self.tfidf_vectorizer is None or self.tfidf_matrix is None:
            print("TF-IDF components not loaded. Search cannot be performed.")
            return []
        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        top_indices = np.argsort(tfidf_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents

class BertSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()

    def encode_batch(self, texts, batch_size=16):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)

    def search(self, query, documents, top_k=5):
        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(documents)
        similarity_scores = cosine_similarity(query_embedding.detach().cpu().numpy(), doc_embeddings.detach().cpu().numpy())[0]
        top_indices = np.argsort(similarity_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents

def read_pdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() for page in reader.pages if page.extract_text())
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from PDF {file_path}: {e}")
        return None

def test_model_time(model, model_name, documents, queries, top_k=3):
    print(f"\n--- {model_name} Time Test ---")
    total_time = 0
    num_queries = len(queries)

    for query in queries:
        start_time = time.time()
        if model_name == "Hybrid Search":
            model.hybrid_search(query, documents, top_k=top_k, tfidf_weight=0.5)
        elif model_name == "TF-IDF Only":
            model.search(query, documents, top_k=top_k)
        elif model_name == "BERT Only":
            model.search(query, documents, top_k=top_k)
        else:
            raise ValueError(f"Unknown model name: {model_name}")
        end_time = time.time()
        search_time = end_time - start_time
        total_time += search_time
        print(f"Query: '{query}' - Search Time: {search_time:.4f} seconds")

    average_time = total_time / num_queries if num_queries > 0 else 0
    print(f"\n--- {model_name} Time Test Summary ---")
    print(f"Average Search Time: {average_time:.4f} seconds")
    return average_time

if __name__ == "__main__":
    # Path to folder containing document files
    folder_path = "/content/drive/MyDrive/documents/"  # Replace with your folder path
    file_names = os.listdir(folder_path)

    # Load all documents from the folder
    documents = []
    doc_names = []
    for file_name in file_names:
        file_path = os.path.join(folder_path, file_name)
        try:
            if file_name.lower().endswith(".pdf"):
                content = read_pdf(file_path)
            else:
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']
                    if not encoding:
                        raise UnicodeDecodeError(f"Encoding not detected for {file_name}.")
                with open(file_path, 'r', encoding=encoding) as file:
                    content = file.read()
            if content:
                documents.append(content)
                doc_names.append(file_name)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

    print(f"Loaded {len(documents)} documents.")

    # Initialize models - loading TF-IDF from pkl files
    hybrid_model = HybridSearchModel()
    tfidf_model = TfidfSearchModel()
    bert_model = BertSearchModel()

    # Define your evaluation queries
    queries = [
        "what is the price?",
        "machine learning applications",
        "cloud computing benefits",
        "artificial intelligence news",
        "deep learning vs neural networks"
    ]

    # Test the time for each model
    test_model_time(hybrid_model, "Hybrid Search", documents, queries)
    test_model_time(tfidf_model, "TF-IDF Only", documents, queries)
    test_model_time(bert_model, "BERT Only", documents, queries)

Loaded 8 documents.
Hybrid Search TF-IDF files not found. Please run the training script first.
TF-IDF Only files not found. Please run the training script first.

--- Hybrid Search Time Test ---
TF-IDF components not loaded. Hybrid search cannot be performed.
Query: 'what is the price?' - Search Time: 0.0000 seconds
TF-IDF components not loaded. Hybrid search cannot be performed.
Query: 'machine learning applications' - Search Time: 0.0000 seconds
TF-IDF components not loaded. Hybrid search cannot be performed.
Query: 'cloud computing benefits' - Search Time: 0.0000 seconds
TF-IDF components not loaded. Hybrid search cannot be performed.
Query: 'artificial intelligence news' - Search Time: 0.0000 seconds
TF-IDF components not loaded. Hybrid search cannot be performed.
Query: 'deep learning vs neural networks' - Search Time: 0.0000 seconds

--- Hybrid Search Time Test Summary ---
Average Search Time: 0.0000 seconds

--- TF-IDF Only Time Test ---
TF-IDF components not loaded. Search can

In [11]:
!pip install torch transformers scikit-learn chardet PyPDF2

import os
import pickle
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import chardet
from PyPDF2 import PdfReader  # PDF handling

class HybridSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased", tfidf_vectorizer_path="tfidf_vectorizer.pkl", tfidf_matrix_path="tfidf_matrix.pkl"):
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        self.tfidf_vectorizer_path = tfidf_vectorizer_path
        self.tfidf_matrix_path = tfidf_matrix_path
        self._load_tfidf()

    def _load_tfidf(self):
        try:
            with open(self.tfidf_vectorizer_path, "rb") as f:
                self.tfidf_vectorizer = pickle.load(f)
            with open(self.tfidf_matrix_path, "rb") as f:
                self.tfidf_matrix = pickle.load(f)
            print("Hybrid Search TF-IDF vectorizer and matrix loaded.")
        except FileNotFoundError:
            print("Hybrid Search TF-IDF files not found. Please run the training script first.")
            self.tfidf_vectorizer = None
            self.tfidf_matrix = None

    def encode_batch(self, texts, batch_size=16):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)

    def hybrid_search(self, query, documents, top_k=5, tfidf_weight=0.4):
        if self.tfidf_vectorizer is None or self.tfidf_matrix is None:
            print("TF-IDF components not loaded. Hybrid search cannot be performed.")
            return []
        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        tfidf_top_indices = np.argsort(tfidf_scores)[-2 * top_k:][::-1]
        filtered_docs = [documents[i] for i in tfidf_top_indices]

        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(filtered_docs)
        bert_scores = torch.nn.functional.cosine_similarity(query_embedding, doc_embeddings).cpu().detach().numpy()

        combined_scores = (
            tfidf_weight * tfidf_scores[tfidf_top_indices] + (1 - tfidf_weight) * bert_scores
        )
        final_indices = np.argsort(combined_scores)[-top_k:][::-1]
        final_ranked_docs = [filtered_docs[i] for i in final_indices]
        return final_ranked_docs

class TfidfSearchModel:
    def __init__(self, vectorizer_path="tfidf_vectorizer_only.pkl", matrix_path="tfidf_matrix_only.pkl"):
        self.tfidf_vectorizer = None
        self.tfidf_matrix = None
        self.vectorizer_path = vectorizer_path
        self.matrix_path = matrix_path
        self._load_tfidf()

    def _load_tfidf(self):
        try:
            with open(self.vectorizer_path, "rb") as f:
                self.tfidf_vectorizer = pickle.load(f)
            with open(self.matrix_path, "rb") as f:
                self.tfidf_matrix = pickle.load(f)
            print("TF-IDF Only vectorizer and matrix loaded.")
        except FileNotFoundError:
            print("TF-IDF Only files not found. Please run the training script first.")
            self.tfidf_vectorizer = None
            self.tfidf_matrix = None

    def search(self, query, documents, top_k=5):
        if self.tfidf_vectorizer is None or self.tfidf_matrix is None:
            print("TF-IDF components not loaded. Search cannot be performed.")
            return []
        query_vector = self.tfidf_vectorizer.transform([query])
        tfidf_scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        top_indices = np.argsort(tfidf_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents

class BertSearchModel:
    def __init__(self, bert_model_name="bert-base-uncased"):
        self.tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        self.bert_model = AutoModel.from_pretrained(bert_model_name).cuda()

    def encode_batch(self, texts, batch_size=16):
        all_embeddings = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
            outputs = self.bert_model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        return torch.nn.functional.normalize(all_embeddings, p=2, dim=1)

    def search(self, query, documents, top_k=5):
        query_embedding = self.encode_batch([query])
        doc_embeddings = self.encode_batch(documents)
        similarity_scores = cosine_similarity(query_embedding.detach().cpu().numpy(), doc_embeddings.detach().cpu().numpy())[0]
        top_indices = np.argsort(similarity_scores)[-top_k:][::-1]
        ranked_documents = [documents[i] for i in top_indices]
        return ranked_documents

def read_pdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = "".join(page.extract_text() for page in reader.pages if page.extract_text())
        return text.strip()
    except Exception as e:
        print(f"Error extracting text from PDF {file_path}: {e}")
        return None

def test_model_time(model, model_name, documents, queries, top_k=3):
    print(f"\n--- {model_name} Time Test ---")
    total_time = 0
    num_queries = len(queries)

    for query in queries:
        start_time = time.time()
        if model_name == "Hybrid Search":
            model.hybrid_search(query, documents, top_k=top_k, tfidf_weight=0.5)
        elif model_name == "TF-IDF Only":
            model.search(query, documents, top_k=top_k)
        elif model_name == "BERT Only":
            model.search(query, documents, top_k=top_k)
        else:
            raise ValueError(f"Unknown model name: {model_name}")
        end_time = time.time()
        search_time = end_time - start_time
        total_time += search_time
        print(f"Query: '{query}' - Search Time: {search_time:.4f} seconds")

    average_time = total_time / num_queries if num_queries > 0 else 0
    print(f"\n--- {model_name} Time Test Summary ---")
    print(f"Average Search Time: {average_time:.4f} seconds")
    return average_time

if __name__ == "__main__":
    # Path to folder containing document files
    folder_path = "/content/drive/MyDrive/documents/"  # Replace with your folder path
    file_names = os.listdir(folder_path)

    # Load all documents from the folder
    documents = []
    doc_names = []
    for file_name in file_names:
        file_path = os.path.join(folder_path, file_name)
        try:
            if file_name.lower().endswith(".pdf"):
                content = read_pdf(file_path)
            else:
                with open(file_path, 'rb') as f:
                    raw_data = f.read()
                    result = chardet.detect(raw_data)
                    encoding = result['encoding']
                    if not encoding:
                        raise UnicodeDecodeError(f"Encoding not detected for {file_name}.")
                with open(file_path, 'r', encoding=encoding) as file:
                    content = file.read()
            if content:
                documents.append(content)
                doc_names.append(file_name)
        except Exception as e:
            print(f"Error loading {file_name}: {e}")
            continue

    print(f"Loaded {len(documents)} documents.")

    # Initialize models - loading TF-IDF from pkl files
    hybrid_model = HybridSearchModel()
    tfidf_model = TfidfSearchModel()
    bert_model = BertSearchModel()

    # Define your evaluation queries
    queries = [
        "what is the price?",
        "machine learning applications",
        "cloud computing benefits",
        "artificial intelligence news",
        "deep learning vs neural networks"
    ]

    # Test the time for each model
    test_model_time(hybrid_model, "Hybrid Search", documents, queries)
    test_model_time(tfidf_model, "TF-IDF Only", documents, queries)
    test_model_time(bert_model, "BERT Only", documents, queries)

Loaded 8 documents.
Hybrid Search TF-IDF vectorizer and matrix loaded.
TF-IDF Only vectorizer and matrix loaded.

--- Hybrid Search Time Test ---
Query: 'what is the price?' - Search Time: 0.5321 seconds
Query: 'machine learning applications' - Search Time: 0.6148 seconds
Query: 'cloud computing benefits' - Search Time: 0.7250 seconds
Query: 'artificial intelligence news' - Search Time: 0.6394 seconds
Query: 'deep learning vs neural networks' - Search Time: 0.6527 seconds

--- Hybrid Search Time Test Summary ---
Average Search Time: 0.6328 seconds

--- TF-IDF Only Time Test ---
Query: 'what is the price?' - Search Time: 0.0027 seconds
Query: 'machine learning applications' - Search Time: 0.0020 seconds
Query: 'cloud computing benefits' - Search Time: 0.0019 seconds
Query: 'artificial intelligence news' - Search Time: 0.0016 seconds
Query: 'deep learning vs neural networks' - Search Time: 0.0016 seconds

--- TF-IDF Only Time Test Summary ---
Average Search Time: 0.0020 seconds

--- BERT